# **Modelo Preditivo Imobiliário**

### Precificação de imóveis residenciais em King County

---

<table width="100%" cellpadding="12">
  <tr><th width="27%" align="left"><big>Identificação</big></th><th align="left"><big>Informação</big></th></tr>
  <tr><td><big><b>Curso</b></big></td><td><big>Desenvolvimento em IA para Análise Preditiva</big></td></tr>
  <tr><td><big><b>Projeto</b></big></td><td><big>Projeto Avaliativo Final — Módulo I</big></td></tr>
  <tr><td><big><b>Professor</b></big></td><td><big>Maurício Costa Cordeiro</big></td></tr>
  <tr><td><big><b>Aluno</b></big></td><td><big>Ricardo Pasquali dos Reis</big></td></tr>
  <tr><td><big><b>Dataset</b></big></td><td><big>King County House Sales — Opção A</big></td></tr>
  <tr><td><big><b>Ambiente</b></big></td><td><big>Google Colab</big></td></tr>
  <tr><td><big><b>Problema</b></big></td><td><big>Regressão supervisionada</big></td></tr>
  <tr><td><big><b>Variável-alvo</b></big></td><td><big><code>price</code>, preço de venda em dólares</big></td></tr>
</table>

---

## **Problema preditivo declarado**

Este projeto busca **prever a variável-alvo `price`**, um valor numérico contínuo que representa o preço de venda do imóvel em dólares, a partir de características físicas, temporais e de localização conhecidas antes da transação.

Isso importa para o negócio porque uma estimativa inicial consistente permite **priorizar imóveis, selecionar comparáveis e identificar preços possivelmente superavaliados ou subavaliados**. Uma diferença de dezenas de milhares de dólares afeta diretamente negociação, entrada, financiamento e exposição ao risco; por isso, o resultado será apresentado com métricas de erro e limites de uso, e não como um preço final automático.

> **Objetivo operacional:** produzir uma referência inicial de preço para apoiar a triagem imobiliária. A previsão não substitui vistoria, análise de comparáveis ou laudo, especialmente nos imóveis de maior valor.


# **1. Preparação do ambiente acadêmico**

A preparação limita-se às importações, às duas dependências geoespaciais, à definição dos caminhos e ao download validado da base pública. As funções analíticas são apresentadas somente nas fases em que passam a ser utilizadas.


In [ ]:
import hashlib, importlib.util, json, os, platform, shutil, subprocess, sys, tempfile, urllib.request
from pathlib import Path

faltantes = [pacote for modulo, pacote in {"h3": "h3>=4.1", "plotly": "plotly>=5.18", "jinja2": "jinja2>=3.1"}.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)

import h3, joblib, matplotlib, matplotlib.pyplot as plt, numpy as np, pandas as pd, plotly
import scipy, seaborn as sns, sklearn
from IPython.display import FileLink, display
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

SEED, TEST_SIZE, CV_FOLDS, N_JOBS, TARGET = 42, 0.20, 5, 1, "price"
em_colab = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path(os.environ["PROJETO_ROOT"]) if os.environ.get("PROJETO_ROOT") else (Path("/content/modelo_preditivo_imobiliario") if em_colab else Path(tempfile.gettempdir()) / "modelo_preditivo_imobiliario")
RAW_DATA = ROOT / "data/raw/kc_house_data.csv"
PROCESSED_DATA, FINAL_DATA = ROOT / "data/processed/kc_house_processed.csv", ROOT / "data/final/kc_house_final.csv"
FIGURES_DIR, MAPS_DIR, MODEL_DIR = ROOT / "outputs/figures", ROOT / "outputs/maps", ROOT / "models/v1"
for pasta in [RAW_DATA.parent, PROCESSED_DATA.parent, FINAL_DATA.parent, FIGURES_DIR, MAPS_DIR, MODEL_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

def baixar_base():
    assinatura = "aa501a2b18e0d0a288a0fdeb043b2c89d6c7e2ead1f9588165537d4e4e12b9df"
    if RAW_DATA.exists() and hashlib.sha256(RAW_DATA.read_bytes()).hexdigest() == assinatura:
        return
    erros = []
    for url in [
        "https://drive.usercontent.google.com/download?id=1XMw1aKYqfFA13AbOzd9zRElv216bCxqZ&export=download&confirm=t",
        "https://drive.google.com/uc?export=download&id=1XMw1aKYqfFA13AbOzd9zRElv216bCxqZ",
    ]:
        parcial = RAW_DATA.with_suffix(".part")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as origem, parcial.open("wb") as destino:
                shutil.copyfileobj(origem, destino, 1024 * 1024)
            obtida = hashlib.sha256(parcial.read_bytes()).hexdigest()
            if obtida != assinatura:
                raise RuntimeError(f"SHA-256 inesperado: {obtida}")
            parcial.replace(RAW_DATA)
            return
        except Exception as erro:
            parcial.unlink(missing_ok=True)
            erros.append(f"{type(erro).__name__}: {erro}")
    raise RuntimeError("Falha no download da base pública: " + " | ".join(erros))

baixar_base()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
pd.set_option("display.max_columns", 30)
{"Python": platform.python_version(), "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "H3": h3.__version__, "Plotly": plotly.__version__}


> **Reprodutibilidade:** o notebook não depende de ZIP, repositório remoto ou módulos auxiliares. O CSV público é conferido por SHA-256; as funções permanecem visíveis junto às respectivas etapas e as versões do ambiente são registradas ao final.


# **2. Fase 1 — Análise Exploratória de Dados (EDA)**

A leitura inicial verifica dimensão, tipos, ausências, duplicidades, distribuição do preço e representatividade temporal antes de qualquer transformação.


In [ ]:
def carregar_dados(caminho=RAW_DATA) -> pd.DataFrame:
    return pd.read_csv(caminho)

def salvar_csv(df: pd.DataFrame, caminho) -> None:
    caminho.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(caminho, index=False)

def _quartos_incompativeis_com_area(df: pd.DataFrame) -> pd.Series:
    area_por_quarto = df["sqft_living"].div(df["bedrooms"].replace(0, np.nan))
    return df["bedrooms"].gt(10) & area_por_quarto.lt(100)

def auditar_dados(df: pd.DataFrame) -> dict:
    contagem_id = df.groupby("id").size()
    ano_venda = pd.to_numeric(df["date"].astype(str).str[:4], errors="coerce")
    return {
        "linhas": int(len(df)),
        "colunas": int(df.shape[1]),
        "duplicatas_exatas": int(df.duplicated().sum()),
        "ids_com_revenda": int(contagem_id.gt(1).sum()),
        "registros_adicionais_de_revenda": int(df.duplicated("id").sum()),
        "ausentes": {k: int(v) for k, v in df.isna().sum().items() if v},
        "quartos_incompativeis_com_area": int(_quartos_incompativeis_com_area(df).sum()),
        "vendas_antes_do_ano_de_construcao": int(df["yr_built"].gt(ano_venda).sum()),
    }

df_raw = carregar_dados()
auditoria = auditar_dados(df_raw)
display(pd.Series(auditoria, name="resultado").to_frame())
display(df_raw.head())
display(df_raw.dtypes.rename("tipo").to_frame())
display(df_raw.describe().T.style.format("{:,.2f}"))


A base reúne **21.613 vendas e 21 colunas**. O preço mediano é **US\$ 450 mil**, abaixo da média de aproximadamente **US\$ 540 mil**, primeira evidência da cauda à direita. Não há duplicatas exatas; IDs repetidos representam revendas e, por isso, serão mantidos e agrupados na separação dos dados.

## **2.1 Escopo e representatividade**

| Dimensão | Cobertura observada | Limite para uso |
|---|---|---|
| temporal | vendas registradas em 2014–2015 | não descreve inflação, juros ou preferências atuais |
| geográfica | King County, Washington | não deve ser transferido automaticamente para outros mercados |
| população | imóveis efetivamente vendidos | não representa anúncios retirados, imóveis sem venda ou todo o estoque residencial |
| finalidade | referência inicial de preço | não substitui vistoria, comparáveis recentes ou laudo |

O projeto é, portanto, um **benchmark histórico e acadêmico**. A validação mede generalização dentro dessa população observada; ela não comprova validade temporal para o mercado atual.


## **2.2 Visualizações e leitura espacial**

As saídas foram separadas em três blocos para que cada evidência tenha sua própria área de visualização no Colab. O primeiro compara distribuição e relações com o preço; o segundo reúne a matriz e as correlações; o terceiro concentra o mapa e o painel geoespacial.


In [ ]:
#@title Rotinas dos gráficos e correlações — apoio
from pathlib import Path
CORES = {"azul": "#1d4ed8", "laranja": "#ea580c", "verde": "#15803d"}
def _salvar(fig, nome: str, pasta: Path = FIGURES_DIR):
    pasta.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(pasta / nome, dpi=160, bbox_inches="tight")
    return fig

def plot_distribuicao_preco(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    sns.histplot(df["price"], bins=55, ax=axes[0], color=CORES["azul"])
    axes[0].set(title="Distribuição do preço", xlabel="Preço (US$)", ylabel="Imóveis")
    sns.histplot(np.log1p(df["price"]), bins=55, ax=axes[1], color=CORES["verde"])
    axes[1].set(title="Distribuição após log1p", xlabel="log1p(preço)", ylabel="Imóveis")
    return _salvar(fig, "01_distribuicao_preco.png")

def plot_relacoes_preco(df: pd.DataFrame):
    amostra = df.sample(min(5_000, len(df)), random_state=42)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    sns.scatterplot(data=amostra, x="sqft_living", y="price", alpha=0.35, ax=axes[0])
    axes[0].set(title="Área construída × preço", xlabel="Área construída (ft²)", ylabel="Preço (US$)")
    sns.boxplot(data=df, x="grade", y="price", color="#93c5fd", showfliers=False, ax=axes[1])
    axes[1].set(title="Padrão construtivo × preço", xlabel="Grade", ylabel="Preço (US$)")
    return _salvar(fig, "02_relacoes_preco.png")

def plot_correlacao(df: pd.DataFrame):
    colunas = [
        "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
        "waterfront", "view", "condition", "grade", "sqft_above", "sqft_basement",
        "yr_built", "lat", "long", "sqft_living15", "sqft_lot15",
    ]
    fig, ax = plt.subplots(figsize=(12.5, 9.5))
    sns.heatmap(df[colunas].corr(), cmap="vlag", center=0, square=True, ax=ax)
    ax.set_title("Correlação de Pearson entre variáveis numéricas")
    return _salvar(fig, "03_correlacao_pearson.png")

def comparar_correlacoes(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    base = df[colunas + [TARGET]]
    resultado = pd.DataFrame(
        {
            "Pearson": base.corr(method="pearson")[TARGET].drop(TARGET),
            "Spearman": base.corr(method="spearman")[TARGET].drop(TARGET),
        }
    )
    resultado["diferenca_absoluta"] = resultado["Spearman"].abs() - resultado["Pearson"].abs()
    return resultado.sort_values("Spearman", key=lambda s: s.abs(), ascending=False)

def resumir_distribuicao_alvo(serie: pd.Series) -> pd.DataFrame:
    """Quantifica a cauda do preço antes e depois da transformação logarítmica."""
    escalas = {"price": serie.astype(float), "log1p(price)": np.log1p(serie)}
    return pd.DataFrame([
        {
            "escala": nome,
            "media": valores.mean(),
            "mediana": valores.median(),
            "assimetria": valores.skew(),
            "curtose_excesso": valores.kurt(),
        }
        for nome, valores in escalas.items()
    ])


### **2.2.1 Distribuição e relações com o preço**

O preço é observado na escala original e após `log1p`. A **assimetria** mede o desequilíbrio das caudas: valor próximo de zero indica maior simetria e valores positivos elevados indicam cauda longa à direita. A **curtose excedente** usa zero como referência; valores muito altos sinalizam concentração central combinada com observações extremas.

Essas medidas não funcionam como teste de normalidade. Aqui, elas servem para verificar se a transformação logarítmica realmente reduz a influência desproporcional dos imóveis mais caros antes da comparação dos modelos.


In [ ]:
diagnostico_alvo = resumir_distribuicao_alvo(df_raw[TARGET])
diagnostico_alvo.to_csv(ROOT / "outputs/diagnostico_distribuicao_alvo.csv", index=False)
display(diagnostico_alvo.style.format({"media": "{:,.2f}", "mediana": "{:,.2f}", "assimetria": "{:.3f}", "curtose_excesso": "{:.3f}"}))
plot_distribuicao_preco(df_raw); plt.show()
plot_relacoes_preco(df_raw); plt.show()


Na escala original, a assimetria é **4,024** e a curtose excedente chega a **34,586**. Após `log1p`, elas caem para **0,428** e **0,692**. A redução quantifica o que o histograma sugere: o log aproxima a massa central e impede que poucas transações milionárias dominem o ajuste, sem alterar a ordem relativa dos imóveis.

A área construída apresenta associação crescente com o valor, enquanto `grade` separa faixas de padrão construtivo. Esses sinais justificam confrontar um palpite ingênuo, um baseline linear e modelos capazes de representar curvaturas e interações.


### **2.2.2 Matriz e correlações Pearson × Spearman**

Pearson resume associação linear; Spearman mede associação monotônica pelos postos e é menos dependente da forma linear. Correlação descreve associação entre variáveis, enquanto regressão estima o alvo usando várias variáveis em conjunto — distinção enfatizada na Semana 11 do curso.


In [ ]:
CORR_COLS = ["sqft_living", "grade", "sqft_living15", "bathrooms", "view", "bedrooms", "lat", "waterfront", "idade_imovel", "log_sqft_lot"]
plot_correlacao(df_raw); plt.show()

ano_venda = pd.to_numeric(df_raw["date"].astype(str).str[:4], errors="coerce")
df_corr = df_raw.assign(
    idade_imovel=(ano_venda - df_raw["yr_built"]).clip(lower=0),
    log_sqft_lot=np.log1p(df_raw["sqft_lot"]),
)
correlacoes = comparar_correlacoes(df_corr, CORR_COLS)
correlacoes.to_csv(ROOT / "outputs/correlacoes_pearson_spearman.csv")
display(correlacoes.style.format("{:.4f}"))


**`sqft_living` e `grade` concentram as associações mais fortes com o preço.** Em `lat`, Spearman supera Pearson, sinal de uma relação espacial monotônica que uma reta isolada não representa adequadamente. A matriz também expõe redundância entre medidas de área; por isso, a etapa seguinte usa VIF antes da modelagem.
